In [31]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage

from langgraph.graph.message import add_messages

from langgraph.store.memory import InMemoryStore
from langgraph.store.memory import BaseStore
from langchain_core.runnables import RunnableConfig

load_dotenv()

True

In [32]:
# create the long term memory 
store = InMemoryStore()

user_id = "u1"

user_details = ("user",user_id,"details")

store.put(user_details, "profile_1", {"data" : "Name : Rudra"})
store.put(user_details, "profile_2", {"data" : "Profession : AI learning engineer"})
store.put(user_details, "preference_1", {"data" : "prefers concise answer"})
store.put(user_details, "preference_2", {"data" : "prefers answer always in just max to max 3-4 lines"})
store.put(user_details, "project_1", {"data" : "Building factlens the fact checker application(python based project)"})

In [33]:
# 2) System prompt template 

SYSTEM_PROMPT_TEMPLATE = """You are a helpful assistant with memory capabilities.
If user-specific memory is available, use it to personalize 
your responses based on what you know about the user.

Your goal is to provide relevant, friendly, and tailored 
assistance that reflects the user’s preferences, context, and past interactions.

If the user’s name or relevant personal context is available, always personalize your responses by:
    – Always Address the user by name (e.g., "Sure, Nitish...") when appropriate
    – Referencing known projects, tools, or preferences (e.g., "your MCP  server python based project")
    – Adjusting the tone to feel friendly, natural, and directly aimed at the user

Avoid generic phrasing when personalization is possible. For example, instead of "In TypeScript apps..." 
say "Since your project is built with TypeScript..."

Use personalization especially in:
    – Greetings and transitions
    – Help or guidance tailored to tools and frameworks the user uses
    – Follow-up messages that continue from past context

Always ensure that personalization is based only on known user details and not assumed.

In the end suggest 3 relevant further questions based on the current response and user profile

The user’s memory (which may be empty) is provided as: {user_details_content}
"""

In [34]:
# state

class ChatbotState(TypedDict):

    messages : Annotated[ list[BaseMessage] , add_messages]

In [35]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [36]:
def chat_node(state: ChatbotState, config: RunnableConfig, store: BaseStore):
    
    user_id = config["configurable"]["user_id"]

    # Read-only: fetch user details memory (no writes)
    user_details = ("user", user_id, "details")
    items = store.search(user_details)

    # Convert memory items into a string blob for {user_details_content}
    # Keep it dead simple for teaching.
    if items:
        user_details_content = "\n".join(f"- {it.value.get('data', '')}" for it in items)
    else:
        user_details_content = ""  # prompt says it may be empty

    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(
        user_details_content=user_details_content
    )

    system_msg = SystemMessage(content=system_prompt)

    response = llm.invoke([system_msg] + state["messages"])
    return {"messages": [response]}

In [37]:
graph = StateGraph(ChatbotState)

graph.add_node("chat_node" , chat_node)

graph.add_edge(START, "chat_node")
graph.add_edge("chat_node", END)

chatbot = graph.compile(store=store)

In [40]:
config = {"configurable": {"user_id": "u1"}}

In [39]:
while True:

    user_msg = input("YOU :: ")

    if user_msg.strip().lower() in ["exit" , "quit", "bye"]:
        break

    initial_state = {"messages" : [{"role": "user", "content": user_msg}]}

    final_state = chatbot.invoke(initial_state, config)["messages"][-1].content

    print(f"YOU :: {user_msg}")
    print(f"AI :: {final_state}")

YOU :: explain about ai
AI :: Certainly, Rudra!

AI, or Artificial Intelligence, refers to the simulation of human intelligence in machines programmed to think and learn like humans. It enables systems to perform tasks such as learning, problem-solving, decision-making, and understanding language, much like the logic you're building into FactLens for fact-checking.

Here are 3 further questions you might have:
1.  What are some key subfields within AI relevant to an AI learning engineer?
2.  How do AI learning models typically get trained for tasks like fact-checking?
3.  What are the main differences between narrow AI and general AI?


In [30]:
items = store.search(user_details)

print(items)

[Item(namespace=['user', 'u1', 'details'], key='profile_1', value={'data': 'Name : Rudra'}, created_at='2026-08-04T11:47:31.939665+00:00', updated_at='2026-08-04T11:47:31.939670+00:00', score=None), Item(namespace=['user', 'u1', 'details'], key='profile_2', value={'data': 'Profession : AI learning engineer'}, created_at='2026-08-04T11:47:31.939798+00:00', updated_at='2026-08-04T11:47:31.939800+00:00', score=None), Item(namespace=['user', 'u1', 'details'], key='preference_1', value={'data': 'prefers concise answer'}, created_at='2026-08-04T11:47:31.939926+00:00', updated_at='2026-08-04T11:47:31.939928+00:00', score=None), Item(namespace=['user', 'u1', 'details'], key='preference_2', value={'data': 'prefers answer always in just max to max 3-4 lines'}, created_at='2026-08-04T11:47:31.940036+00:00', updated_at='2026-08-04T11:47:31.940038+00:00', score=None), Item(namespace=['user', 'u1', 'details'], key='project_1', value={'data': 'Building factlens the fact checker application(python bas